# Arabic Contextualized Embeddings — Parallel Version

This notebook generates **Arabic XLM-RoBERTa contextual embeddings** aligned to Eyal's filtered word-level transcript.

The important design choice is that Arabic is aligned through the **English sentence anchor**:

```text
Eyal's timestamped English word
        ↓
find the English sentence it belongs to
        ↓
use the Arabic sentence with the same sentence_id
        ↓
try to extract the Arabic target word in context
        ↓
if Arabic word match fails, use the Arabic sentence embedding as contextual fallback
        ↓
if English sentence assignment fails, use Arabic word-in-isolation fallback
```

This avoids relying only on direct Arabic word matching, which is unstable because Arabic sentence translation may use different morphology, prefixes/suffixes, word order, or vocabulary.

**Outputs:**
- `ar_contextual_aligned_embeddings.csv` — exactly `(1735, 768)` vectors
- `ar_contextual_quality_flags.csv` — metadata and fallback type for each word

## 1. Load Data

In [1]:
import os
import re
import csv
import unicodedata
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

# =========================
# Robust path helpers
# =========================

def first_existing_path(candidates):
    """Return the first path that exists from a list of candidate paths."""
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError("None of these paths exist:\n" + "\n".join(candidates))

# These relative paths assume the notebook is in the notebooks/ directory.
WORD_LEVEL_PATH = first_existing_path([
    r"../data/Amirim_Project_Submission/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv",
    r"../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv",
    r"../data/processed/translated_podcast_transcript_filtered.csv",
    r"../data/translated_podcast_transcript_filtered.csv",
])

EN_SENTENCES_PATH = first_existing_path([
    r"../data/sentences/podcast_sentences_en.csv",
    r"../data/processed/podcast_sentences_en.csv",
    r"../data/podcast_sentences_en.csv",
])

AR_SENTENCES_PATH = first_existing_path([
    r"../data/sentences/podcast_sentences_ar.csv",
    r"../data/processed/podcast_sentences_ar.csv",
    r"../data/podcast_sentences_ar.csv",
])

OUT_EMBEDDINGS = r"../data/processed/ar_contextual_aligned_embeddings.csv"
OUT_FLAGS = r"../data/processed/ar_contextual_quality_flags.csv"
os.makedirs(os.path.dirname(OUT_EMBEDDINGS), exist_ok=True)

print("WORD_LEVEL_PATH:", WORD_LEVEL_PATH)
print("EN_SENTENCES_PATH:", EN_SENTENCES_PATH)
print("AR_SENTENCES_PATH:", AR_SENTENCES_PATH)

# =========================
# Robust sentence CSV loader
# =========================

def load_sentence_csv(path, preferred_col=None):
    """
    Loads sentence files robustly.

    Supports:
    1. sentence_id,sentence
    2. sentence_id,sentence_en,sentence_he
    3. broken CSV rows where commas inside sentences were not quoted
       by splitting only on the first comma.
    """
    # First try pandas for valid CSVs
    try:
        df = pd.read_csv(path)
        if preferred_col and preferred_col in df.columns:
            return pd.DataFrame({
                "sentence_id": df["sentence_id"].astype(int),
                "sentence": df[preferred_col].astype(str)
            })
        if "sentence" in df.columns:
            return pd.DataFrame({
                "sentence_id": df["sentence_id"].astype(int),
                "sentence": df["sentence"].astype(str)
            })
    except Exception:
        pass

    # Fallback: split each line only on the first comma
    rows = []
    with open(path, "r", encoding="utf-8-sig") as f:
        lines = f.readlines()

    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        sentence_id, sentence = line.split(",", 1)
        sentence = sentence.strip()
        if sentence.startswith('"') and sentence.endswith('"'):
            sentence = sentence[1:-1]
        rows.append({"sentence_id": int(sentence_id), "sentence": sentence})

    return pd.DataFrame(rows)

# =========================
# Load datasets
# =========================

print("\nLoading datasets...")
word_level_df = pd.read_csv(WORD_LEVEL_PATH)
en_sentences_df = load_sentence_csv(EN_SENTENCES_PATH)
ar_sentences_df = load_sentence_csv(AR_SENTENCES_PATH)

print("word_level_df:", word_level_df.shape)
print("en_sentences_df:", en_sentences_df.shape)
print("ar_sentences_df:", ar_sentences_df.shape)

print("\nword_level_df columns:", list(word_level_df.columns))
print("en_sentences_df columns:", list(en_sentences_df.columns))
print("ar_sentences_df columns:", list(ar_sentences_df.columns))

assert "en" in word_level_df.columns, "Missing 'en' column in word-level file"
assert "ar" in word_level_df.columns, "Missing 'ar' column in word-level file"
assert len(en_sentences_df) == len(ar_sentences_df), "English and Arabic sentence files must be parallel and same length"

print("\nFirst rows from Eyal word-level file:")
display(word_level_df[["start", "end", "en", "ar"]].head())

print("\nFirst English sentences:")
display(en_sentences_df.head())

print("\nFirst Arabic sentences:")
display(ar_sentences_df.head())

c:\Users\mayat\miniconda3\envs\language_project_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WORD_LEVEL_PATH: ../data/Amirim_Project_Submission/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv
EN_SENTENCES_PATH: ../data/sentences/podcast_sentences_en.csv
AR_SENTENCES_PATH: ../data/sentences/podcast_sentences_ar.csv

Loading datasets...
word_level_df: (1735, 5)
en_sentences_df: (402, 2)
ar_sentences_df: (402, 2)

word_level_df columns: ['start', 'end', 'en', 'he', 'ar']
en_sentences_df columns: ['sentence_id', 'sentence']
ar_sentences_df columns: ['sentence_id', 'sentence']

First rows from Eyal word-level file:


,start,end,en,ar
0,3.710,3.790,act,حملة
1,4.651,4.931,monkey,قرد
2,5.151,5.391,middle,وسط
3,7.072,7.342,places,الأماكن
4,7.542,7.782,animals,الحيوانات



First English sentences:


,sentence_id,sentence
0,1,"Act One, Monkey in the Middle."
1,2,So there's some places where animals almost ne...
2,3,"This act ends up in a place like that, but it ..."
3,4,Dana Chivvis explains.
4,5,Our story begins deep in the rainforests of In...



First Arabic sentences:


,sentence_id,sentence
0,1,الفصل الأول، القرد في المنتصف.
1,2,هناك أماكن لا تكاد تذهب إليها الحيوانات أبداً،...
2,3,ينتهي هذا الفصل في مكان كهذا، لكنه يبدأ بعيداً...
3,4,دانا تشيفيس تشرح.
4,5,تبدأ قصتنا في أعماق غابات إندونيسيا المطيرة عل...


## 2. Matching Helpers

In [2]:
# =========================
# English normalization helpers for sentence assignment
# =========================

def normalize_en(text):
    """Normalize English text for matching only, not for embeddings."""
    text = str(text).lower()
    text = text.replace("_", " ")
    text = text.replace("’", "'")

    # Normalize common contractions / cleaned transcript forms
    replacements = {
        "don't": "dont", "can't": "cant", "won't": "wont",
        "i'm": "im", "it's": "its", "that's": "thats",
        "there's": "theres", "you're": "youre", "we're": "were",
        "they're": "theyre", "didn't": "didnt", "doesn't": "doesnt",
        "couldn't": "couldnt", "wouldn't": "wouldnt", "shouldn't": "shouldnt"
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_en_components(target):
    target = normalize_en(target)
    return [x for x in target.split() if x]


def get_sentence_tokens_en(sentence):
    sentence = normalize_en(sentence)
    return [x for x in sentence.split() if x]


def count_occurrences_en(sentence, components):
    tokens = get_sentence_tokens_en(sentence)
    n = len(components)
    if n == 0 or len(tokens) < n:
        return 0
    count = 0
    for i in range(len(tokens) - n + 1):
        if tokens[i:i+n] == components:
            count += 1
    return count

# =========================
# Arabic normalization helpers for optional Arabic word-level extraction
# =========================

AR_PREFIXES = sorted([
    "وال", "بال", "لل", "كال", "فال",
    "ال", "و", "ب", "ل", "ك", "ف", "س"
], key=len, reverse=True)

AR_SUFFIXES = sorted([
    "تان", "ون", "ين", "ات", "ان",
    "ها", "هم", "هن", "كم", "كن", "نا", "ني",
    "ة", "ي", "ك", "ه"
], key=len, reverse=True)


def normalize_ar(text):
    """
    Normalize Arabic text for matching only.
    The original Arabic sentence is still used for XLM-R embeddings.
    """
    text = unicodedata.normalize("NFC", str(text))

    # Remove Arabic diacritics / tashkeel
    text = "".join(c for c in text if not ("\u064B" <= c <= "\u065F"))

    # Remove tatweel / kashida elongation
    text = text.replace("\u0640", "")

    # Normalize common Arabic letter variants
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و").replace("ئ", "ي")

    # Strip edge punctuation
    text = text.strip(' .,!?:;"\'()-[]{}،؟؛')
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_arabic_stem(token):
    """
    Light Arabic stemming for matching.
    This is not a full morphological analyzer.
    """
    token = normalize_ar(token)
    candidates = {token}

    # Strip prefixes
    prefix_stripped = [token]
    for prefix in AR_PREFIXES:
        if token.startswith(prefix) and len(token) > len(prefix) + 2:
            stripped = token[len(prefix):]
            candidates.add(stripped)
            prefix_stripped.append(stripped)

    # Strip suffixes
    for base in prefix_stripped:
        for suffix in AR_SUFFIXES:
            if base.endswith(suffix) and len(base) > len(suffix) + 2:
                candidates.add(base[:-len(suffix)])

    return [c for c in candidates if c]


def get_target_components_ar(target):
    target = normalize_ar(target)
    components = [x for x in target.split() if x]
    return components


def arabic_components_match(sentence_token, target_component):
    sent_candidates = get_arabic_stem(sentence_token)
    target_candidates = get_arabic_stem(target_component)
    return any(s == t for s in sent_candidates for t in target_candidates)


def get_arabic_word_spans(sentence):
    """
    Return rough whitespace-token spans from the original sentence.
    Each item includes original text, normalized text, start char, end char.
    """
    spans = []
    for m in re.finditer(r"\S+", str(sentence)):
        raw = m.group(0)
        norm = normalize_ar(raw)
        if norm:
            spans.append({
                "text": raw,
                "norm": norm,
                "start": m.start(),
                "end": m.end()
            })
    return spans


def find_arabic_component_span(sentence, target_components, consumed_word_positions=None):
    """
    Try to find Eyal's Arabic target components inside the Arabic sentence.
    Returns a list of word-span indices if found, else None.
    """
    if consumed_word_positions is None:
        consumed_word_positions = set()

    if not target_components:
        return None

    spans = get_arabic_word_spans(sentence)
    n = len(target_components)

    for i in range(len(spans) - n + 1):
        positions = set(range(i, i+n))
        if positions & consumed_word_positions:
            continue

        ok = True
        for j in range(n):
            if not arabic_components_match(spans[i+j]["norm"], target_components[j]):
                ok = False
                break
        if ok:
            return list(range(i, i+n))

    return None

# Precompute target component lists
EN_TARGET_COMPONENTS = [get_en_components(x) for x in word_level_df["en"]]
AR_TARGET_COMPONENTS = [get_target_components_ar(x) for x in word_level_df["ar"]]

print("Example English targets:")
for i in range(5):
    print(i, word_level_df.loc[i, "en"], "->", EN_TARGET_COMPONENTS[i])

print("\nExample Arabic targets:")
for i in range(5):
    print(i, word_level_df.loc[i, "ar"], "->", AR_TARGET_COMPONENTS[i])

Example English targets:
0 act -> ['act']
1 monkey -> ['monkey']
2 middle -> ['middle']
3 places -> ['places']
4 animals -> ['animals']

Example Arabic targets:
0 حملة -> ['حملة']
1 قرد -> ['قرد']
2 وسط -> ['وسط']
3 الأماكن -> ['الاماكن']
4 الحيوانات -> ['الحيوانات']


## 3. Assign Eyal's Words to English Sentences, Then Use Parallel Arabic Sentences

In [3]:
print("Assigning Eyal's English target words to English sentences...")

en_sentences = en_sentences_df["sentence"].tolist()
ar_sentences = ar_sentences_df["sentence"].tolist()

word_to_sentence_en = {}
unassigned_en = []
en_assignment_counts = defaultdict(lambda: defaultdict(int))

en_sent_idx = 0
LOOKAHEAD = 25
n_sentences = len(en_sentences)

for wi in tqdm(range(len(word_level_df)), desc="English sentence assignment"):
    components = EN_TARGET_COMPONENTS[wi]
    target_key = " ".join(components)

    if not components:
        word_to_sentence_en[wi] = None
        unassigned_en.append(wi)
        continue

    found = False
    search_end = min(en_sent_idx + LOOKAHEAD + 1, n_sentences)

    for candidate_idx in range(en_sent_idx, search_end):
        capacity = count_occurrences_en(en_sentences[candidate_idx], components)
        already_assigned = en_assignment_counts[candidate_idx][target_key]

        if capacity > already_assigned:
            word_to_sentence_en[wi] = candidate_idx
            en_assignment_counts[candidate_idx][target_key] += 1

            # Advance pointer conservatively
            if en_assignment_counts[candidate_idx][target_key] >= capacity and candidate_idx > en_sent_idx:
                en_sent_idx = candidate_idx

            found = True
            break

    if not found:
        word_to_sentence_en[wi] = None
        unassigned_en.append(wi)

assigned_en = sum(1 for v in word_to_sentence_en.values() if v is not None)
print("=" * 60)
print("ENGLISH-ANCHOR FIRST-PASS REPORT")
print("=" * 60)
print(f"Total target words : {len(word_level_df)}")
print(f"Assigned           : {assigned_en}")
print(f"Unassigned         : {len(unassigned_en)}")
print(f"Assignment rate    : {assigned_en / len(word_level_df):.1%}")

print("\nFirst 20 unassigned examples:")
for wi in unassigned_en[:20]:
    row = word_level_df.iloc[wi]
    print(f"  [{wi:4d}] en='{row['en']}' | ar='{row['ar']}'")

print("\nFirst 10 assignments:")
shown = 0
for wi, si in word_to_sentence_en.items():
    if si is not None:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] en='{row['en']}' | ar='{row['ar']}' -> sentence {si} / id={ar_sentences_df.loc[si, 'sentence_id']}")
        shown += 1
        if shown >= 10:
            break

Assigning Eyal's English target words to English sentences...


English sentence assignment: 100%|██████████| 1735/1735 [00:00<00:00, 19285.05it/s]

ENGLISH-ANCHOR FIRST-PASS REPORT
Total target words : 1735
Assigned           : 103
Unassigned         : 1632
Assignment rate    : 5.9%

First 20 unassigned examples:
  [  50] en='mean' | ar='متوسط'
  [  53] en='know' | ar='يعرف'
  [  57] en='etiquette' | ar='آداب السلوك'
  [  58] en='showing' | ar='عرض تقديمي'
  [  59] en='teeth' | ar='أسنان'
  [  60] en='show' | ar='عرض'
  [  61] en='teeth' | ar='أسنان'
  [  62] en='stare' | ar='التحديق'
  [  64] en='secrets' | ar='أسرار'
  [  65] en='able' | ar='قادر'
  [  66] en='good' | ar='جيد'
  [  67] en='photography' | ar='التصوير الفوتوغرافي'
  [  68] en='making' | ar='تحضير'
  [  69] en='animal' | ar='حيوان'
  [  70] en='relaxed' | ar='هادئ'
  [  71] en='shot' | ar='ركلة'
  [  73] en='full' | ar='ممتلىء'
  [  74] en='face' | ar='وجه'
  [  75] en='portrait' | ar='لَوحَة'
  [  76] en='lens' | ar='عدسة'

First 10 assignments:
  [   0] en='act' | ar='حملة' -> sentence 0 / id=1
  [   1] en='monkey' | ar='قرد' -> sentence 0 / id=1
  [   2] en='mid

## 3b. Rescue Unassigned English Assignments by Full Scan

In [4]:
print(f"Rescuing {len(unassigned_en)} English-unassigned words via full scan...\n")

# Approximate sentence time map from successfully assigned words
sentence_time_map = {}
for wi, si in word_to_sentence_en.items():
    if si is None:
        continue
    t_start = float(word_level_df.iloc[wi]["start"])
    t_end = float(word_level_df.iloc[wi]["end"])
    if si not in sentence_time_map:
        sentence_time_map[si] = [t_start, t_end]
    else:
        sentence_time_map[si][0] = min(sentence_time_map[si][0], t_start)
        sentence_time_map[si][1] = max(sentence_time_map[si][1], t_end)

rescued = 0
still_unassigned = []

for wi in unassigned_en:
    components = EN_TARGET_COMPONENTS[wi]
    target_key = " ".join(components)
    t = float(word_level_df.iloc[wi]["start"])

    if not components:
        still_unassigned.append(wi)
        continue

    candidates = []
    for si in range(n_sentences):
        capacity = count_occurrences_en(en_sentences[si], components)
        remaining = capacity - en_assignment_counts[si][target_key]
        if remaining > 0:
            if si in sentence_time_map:
                proximity = abs(sentence_time_map[si][0] - t)
            else:
                proximity = 999999
            candidates.append((proximity, si))

    if candidates:
        candidates.sort()
        best_si = candidates[0][1]
        word_to_sentence_en[wi] = best_si
        en_assignment_counts[best_si][target_key] += 1
        rescued += 1
    else:
        still_unassigned.append(wi)

unassigned_en = still_unassigned
assigned_en = sum(1 for v in word_to_sentence_en.values() if v is not None)

print(f"Rescued via full scan : {rescued}")
print(f"Still unassigned      : {len(unassigned_en)}")
print(f"New assignment rate   : {assigned_en / len(word_level_df):.1%}")

if unassigned_en:
    print("\nFirst 30 still unassigned:")
    for wi in unassigned_en[:30]:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] t={row['start']:.1f}s en='{row['en']}' ar='{row['ar']}'")

Rescuing 1632 English-unassigned words via full scan...

Rescued via full scan : 1603
Still unassigned      : 29
New assignment rate   : 98.3%

First 30 still unassigned:
  [ 291] t=270.0s en='last_years' ar='السنوات الأخيرة'
  [ 329] t=307.1s en='wanting' ar='يريد'
  [ 418] t=397.7s en='fair_use' ar='الاستخدام العادل'
  [ 427] t=409.1s en='alls' ar='كل شئ'
  [ 807] t=761.4s en='created' ar='مخلوق'
  [ 894] t=858.1s en='need' ar='يحتاج'
  [ 934] t=910.0s en='cause' ar='سبب'
  [1039] t=1021.8s en='beaching' ar='شاطئ'
  [1089] t=1073.7s en='dash' ar='خطوة'
  [1095] t=1081.6s en='afternoon' ar='ظهراً'
  [1100] t=1085.6s en='unanticipated' ar='غير متوقع'
  [1140] t=1128.1s en='answered' ar='إجابة'
  [1177] t=1163.7s en='fourteenth' ar='أربعة عشر'
  [1195] t=1181.2s en='endgame' ar='نهاية اللعبة'
  [1378] t=1399.8s en='trying' ar='محاولة'
  [1386] t=1404.5s en='cause' ar='سبب'
  [1404] t=1424.8s en='cause' ar='سبب'
  [1408] t=1430.6s en='part' ar='جزء'
  [1447] t=1481.6s en='appeals_court' 

## 3c. Build Parallel Arabic Sentence Assignment

In [5]:
# Use the English sentence assignment to select the parallel Arabic sentence.
# If English sentence assignment failed, this word will use word-isolation fallback later.

word_to_sentence_ar = {}
for wi, en_si in word_to_sentence_en.items():
    word_to_sentence_ar[wi] = en_si if en_si is not None else None

assigned_parallel_ar = sum(1 for v in word_to_sentence_ar.values() if v is not None)

print("=" * 60)
print("PARALLEL ARABIC SENTENCE ASSIGNMENT")
print("=" * 60)
print(f"Arabic sentences assigned via English anchor: {assigned_parallel_ar} / {len(word_level_df)}")
print(f"Parallel assignment rate                 : {assigned_parallel_ar / len(word_level_df):.1%}")

print("\nFirst 10 parallel Arabic assignments:")
shown = 0
for wi, si in word_to_sentence_ar.items():
    if si is not None:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] en='{row['en']}' | ar_target='{row['ar']}' -> ar_sentence_id={ar_sentences_df.loc[si, 'sentence_id']}")
        print(f"       AR sentence: {ar_sentences[si][:100]}")
        shown += 1
        if shown >= 10:
            break

PARALLEL ARABIC SENTENCE ASSIGNMENT
Arabic sentences assigned via English anchor: 1706 / 1735
Parallel assignment rate                 : 98.3%

First 10 parallel Arabic assignments:
  [   0] en='act' | ar_target='حملة' -> ar_sentence_id=1
       AR sentence: الفصل الأول، القرد في المنتصف.
  [   1] en='monkey' | ar_target='قرد' -> ar_sentence_id=1
       AR sentence: الفصل الأول، القرد في المنتصف.
  [   2] en='middle' | ar_target='وسط' -> ar_sentence_id=1
       AR sentence: الفصل الأول، القرد في المنتصف.
  [   3] en='places' | ar_target='الأماكن' -> ar_sentence_id=2
       AR sentence: هناك أماكن لا تكاد تذهب إليها الحيوانات أبداً، أماكن صُممت من قبل البشر وللبشر.
  [   4] en='animals' | ar_target='الحيوانات' -> ar_sentence_id=2
       AR sentence: هناك أماكن لا تكاد تذهب إليها الحيوانات أبداً، أماكن صُممت من قبل البشر وللبشر.
  [   5] en='go' | ar_target='يذهب' -> ar_sentence_id=2
       AR sentence: هناك أماكن لا تكاد تذهب إليها الحيوانات أبداً، أماكن صُممت من قبل البشر وللبشر.
  [  

## 4. Load XLM-RoBERTa

In [6]:
print("Loading XLM-RoBERTa...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModel.from_pretrained("xlm-roberta-base").to(device)
model.eval()

print("Model ready.")

Loading XLM-RoBERTa...
Device: cpu
Model ready.


## 5. Embedding Helpers

In [7]:
def token_vectors_with_offsets(text):
    """
    Run XLM-R on text and return token offsets + vectors.
    Offsets are character offsets into the original text.
    """
    encoded = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=512
    )
    offsets = encoded.pop("offset_mapping")[0].tolist()
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        output = model(**encoded)

    vectors = output.last_hidden_state[0].detach().cpu().numpy()
    input_ids = encoded["input_ids"][0].detach().cpu().tolist()
    special_ids = set(tokenizer.all_special_ids)

    token_info = []
    for idx, (start, end) in enumerate(offsets):
        if input_ids[idx] in special_ids:
            continue
        if start == end:
            continue
        token_info.append({
            "token_idx": idx,
            "start": start,
            "end": end,
            "vector": vectors[idx]
        })

    return token_info


def mean_pool_text(text):
    """Mean-pool all non-special XLM-R token vectors for a text."""
    token_info = token_vectors_with_offsets(str(text))
    if not token_info:
        return np.zeros(768, dtype=float)
    return np.mean([x["vector"] for x in token_info], axis=0)


def vector_for_char_spans(text, char_spans):
    """
    Return mean vector for XLM-R tokens overlapping given char spans.
    char_spans is a list of (start, end) pairs.
    """
    token_info = token_vectors_with_offsets(str(text))
    selected_vectors = []

    for tok in token_info:
        tok_start, tok_end = tok["start"], tok["end"]
        for span_start, span_end in char_spans:
            # overlap condition
            if tok_start < span_end and tok_end > span_start:
                selected_vectors.append(tok["vector"])
                break

    if selected_vectors:
        return np.mean(selected_vectors, axis=0)
    return None


def get_arabic_word_context_vector(sentence, target_components, consumed_word_positions=None):
    """
    Try to find Eyal's Arabic target in the Arabic sentence and extract
    its contextual vector. Returns (vector, matched_word_positions).
    """
    if consumed_word_positions is None:
        consumed_word_positions = set()

    matched_positions = find_arabic_component_span(
        sentence,
        target_components,
        consumed_word_positions=consumed_word_positions
    )

    if matched_positions is None:
        return None, None

    word_spans = get_arabic_word_spans(sentence)
    char_spans = [(word_spans[pos]["start"], word_spans[pos]["end"]) for pos in matched_positions]
    vec = vector_for_char_spans(sentence, char_spans)

    if vec is None:
        return None, None

    return vec, matched_positions

# Caches to avoid re-running model too much
sentence_mean_cache = {}
word_isolation_cache = {}

def get_sentence_context_fallback(si):
    if si not in sentence_mean_cache:
        sentence_mean_cache[si] = mean_pool_text(ar_sentences[si])
    return sentence_mean_cache[si]


def get_word_isolation_fallback(raw_ar):
    raw_ar = str(raw_ar).strip()
    if raw_ar not in word_isolation_cache:
        word_isolation_cache[raw_ar] = mean_pool_text(raw_ar)
    return word_isolation_cache[raw_ar]

## 6. Extraction Loop

In [8]:
print("Extracting Arabic contextual embeddings...\n")

ar_final_embeddings = [None] * len(word_level_df)
ar_fallback_type = [None] * len(word_level_df)
ar_assigned_sentence_id = [None] * len(word_level_df)
ar_word_context_matches = []
ar_sentence_context_fallbacks = []
ar_word_isolation_fallbacks = []

# Group by Arabic sentence assignment
words_per_sentence = defaultdict(list)
for wi, si in word_to_sentence_ar.items():
    if si is not None:
        words_per_sentence[si].append(wi)

# Process words with a parallel Arabic sentence
for si in tqdm(sorted(words_per_sentence.keys()), desc="Processing Arabic sentences"):
    consumed_positions = set()
    for wi in sorted(words_per_sentence[si]):
        raw_ar = str(word_level_df.iloc[wi]["ar"]).strip()
        target_components = AR_TARGET_COMPONENTS[wi]
        sentence = ar_sentences[si]

        vec, matched_positions = get_arabic_word_context_vector(
            sentence,
            target_components,
            consumed_word_positions=consumed_positions
        )

        if vec is not None:
            ar_final_embeddings[wi] = vec
            ar_fallback_type[wi] = "full_word_context"
            ar_word_context_matches.append(wi)
            consumed_positions.update(matched_positions)
        else:
            # Arabic target not found in parallel Arabic sentence.
            # Still use the sentence as contextual representation.
            ar_final_embeddings[wi] = get_sentence_context_fallback(si)
            ar_fallback_type[wi] = "sentence_context_fallback"
            ar_sentence_context_fallbacks.append(wi)

        ar_assigned_sentence_id[wi] = int(ar_sentences_df.iloc[si]["sentence_id"])

# Process words with no English/parallel sentence assignment
for wi in range(len(word_level_df)):
    if ar_final_embeddings[wi] is None:
        raw_ar = str(word_level_df.iloc[wi]["ar"]).strip()
        ar_final_embeddings[wi] = get_word_isolation_fallback(raw_ar)
        ar_fallback_type[wi] = "word_isolation_fallback"
        ar_word_isolation_fallbacks.append(wi)
        ar_assigned_sentence_id[wi] = None

# Verify no missing vectors
assert all(vec is not None for vec in ar_final_embeddings), "Some Arabic embeddings are missing"

ar_embeddings_matrix = np.vstack(ar_final_embeddings)
print("Extraction complete.")
print("Arabic embeddings matrix:", ar_embeddings_matrix.shape)

Extracting Arabic contextual embeddings...



Processing Arabic sentences: 100%|██████████| 385/385 [01:26<00:00,  4.47it/s]


Extraction complete.
Arabic embeddings matrix: (1735, 768)


## 7. Report

In [9]:
print("=" * 60)
print("ARABIC CONTEXTUAL EMBEDDING REPORT")
print("=" * 60)
print(f"Total target words              : {len(word_level_df)}")
print(f"Full Arabic word-context matches: {len(ar_word_context_matches)}")
print(f"Sentence-context fallbacks      : {len(ar_sentence_context_fallbacks)}")
print(f"Word-isolation fallbacks        : {len(ar_word_isolation_fallbacks)}")
print(f"Total vectors                   : {len(ar_final_embeddings)}")

full_rate = len(ar_word_context_matches) / len(word_level_df) * 100
sent_context_rate = (len(ar_word_context_matches) + len(ar_sentence_context_fallbacks)) / len(word_level_df) * 100
print(f"Full word-context rate          : {full_rate:.1f}%")
print(f"Any contextual sentence rate    : {sent_context_rate:.1f}%")

assert ar_embeddings_matrix.shape == (len(word_level_df), 768), \
    f"Expected {(len(word_level_df), 768)}, got {ar_embeddings_matrix.shape}"

print("\nVerified: every target word has exactly one 768-dimensional vector.")

print("\nFirst 20 sentence-context fallbacks:")
for wi in ar_sentence_context_fallbacks[:20]:
    row = word_level_df.iloc[wi]
    print(f"  [{wi:4d}] en='{row['en']}' | ar='{row['ar']}' | sentence_id={ar_assigned_sentence_id[wi]}")

print("\nFirst 20 word-isolation fallbacks:")
for wi in ar_word_isolation_fallbacks[:20]:
    row = word_level_df.iloc[wi]
    print(f"  [{wi:4d}] en='{row['en']}' | ar='{row['ar']}'")

ARABIC CONTEXTUAL EMBEDDING REPORT
Total target words              : 1735
Full Arabic word-context matches: 570
Sentence-context fallbacks      : 1136
Word-isolation fallbacks        : 29
Total vectors                   : 1735
Full word-context rate          : 32.9%
Any contextual sentence rate    : 98.3%

Verified: every target word has exactly one 768-dimensional vector.

First 20 sentence-context fallbacks:
  [   0] en='act' | ar='حملة' | sentence_id=1
  [   2] en='middle' | ar='وسط' | sentence_id=1
  [   5] en='go' | ar='يذهب' | sentence_id=2
  [   7] en='designed' | ar='مصمم' | sentence_id=2
  [   8] en='humans' | ar='الجنس البشري' | sentence_id=2
  [   9] en='humans' | ar='الجنس البشري' | sentence_id=2
  [  10] en='act' | ar='حملة' | sentence_id=3
  [  11] en='ends' | ar='نهاية' | sentence_id=3
  [  13] en='starts' | ar='بدء' | sentence_id=3
  [  14] en='explains' | ar='يشرح' | sentence_id=4
  [  15] en='story' | ar='قصة' | sentence_id=5
  [  16] en='begins' | ar='بدء' | sentence

## 8. Save Outputs

In [ ]:
# Save embeddings: 1735 rows x 768 columns
ar_embeddings_df = pd.DataFrame(ar_embeddings_matrix)
ar_embeddings_df.to_csv(OUT_EMBEDDINGS, index=False)

# Save quality flags and metadata
ar_quality_df = pd.DataFrame({
    "word_idx": list(range(len(word_level_df))),
    "start": word_level_df["start"].tolist(),
    "end": word_level_df["end"].tolist(),
    "en": word_level_df["en"].tolist(),
    "ar": word_level_df["ar"].tolist(),
    "assigned_sentence_id": ar_assigned_sentence_id,
    "fallback_type": ar_fallback_type,
    "is_full_word_context": [x == "full_word_context" for x in ar_fallback_type],
    "is_sentence_context_fallback": [x == "sentence_context_fallback" for x in ar_fallback_type],
    "is_word_isolation_fallback": [x == "word_isolation_fallback" for x in ar_fallback_type],
})
ar_quality_df.to_csv(OUT_FLAGS, index=False, encoding="utf-8-sig")

print(f"Saved embeddings -> {OUT_EMBEDDINGS}")
print(f"Shape            -> {ar_embeddings_df.shape}")
print(f"Any NaN          -> {ar_embeddings_df.isnull().any().any()}")
print(f"Saved flags      -> {OUT_FLAGS}")

display(ar_quality_df.head())
print("\nSample embedding row 0, first 5 dims:")
print(ar_embeddings_df.iloc[0, :5].tolist())